In [1]:
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv(override=True)
gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY not found in .env file. Add it before continuing.")

# ── Initialise the Gemini LLM  ─────────────────────
gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    max_output_tokens=512,
    google_api_key=gemini_api_key,
)

In [2]:
from langchain_classic.agents import Tool, initialize_agent, AgentType
print("🤖 SECTION 1: Agents & Tools with Gemini")
print("=" * 55)

# ── Define custom tool functions ──────────────────────────────────────────
def calculator(expression: str) -> str:
    """Safely evaluate a simple arithmetic expression."""
    try:
        allowed = set("0123456789+-*/()., ")
        if not all(c in allowed for c in expression):
            return "Error: Only arithmetic expressions are allowed."
        result = eval(expression)   # safe for classroom demo
        return f"{expression} = {result}"
    except Exception as e:
        return f"Calculation error: {e}"

def word_counter(text: str) -> str:
    """Count the number of words in the given text."""
    count = len(text.split())
    return f"The text has {count} word(s)."

def reverse_text(text: str) -> str:
    """Reverse the given string."""
    return text[::-1]


# ── Wrap functions as LangChain Tools ────────────────────────────────────
tools = [
    Tool(
        name="calculator",
        func=calculator,
        description="Useful for evaluating arithmetic expressions. Input should be a math expression like '25 * 4 + 10'.",
    ),
    Tool(
        name="word_counter",
        func=word_counter,
        description="Counts words in a sentence. Input is any text string.",
    ),
    Tool(
        name="reverse_text",
        func=reverse_text,
        description="Reverses a given string. Input is any text.",
    ),
]

# ── Initialise ReAct agent ────────────────────────────────────────────────
agent = initialize_agent(
    tools=tools,
    llm=gemini_llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True,
)
print("\n─── Agent Test 1: Calculator ───")
try:
    result = agent.run("What is 125 * 8 + 37?")
    print(f"✅ Answer: {result}")
except Exception as e:
    print(f"❌ {e}")

🤖 SECTION 1: Agents & Tools with Gemini

─── Agent Test 1: Calculator ───


> Entering new AgentExecutor chain...


C:\Users\Roni\AppData\Local\Temp\ipykernel_12712\2117704194.py:47: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-built ReAct agent](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/).
  agent = initialize_agent(
C:\Users\Roni\AppData\Local\Temp\ipykernel_12712\2117704194.py:57: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = agent.run("What is 125 * 8 + 37?")


Action: calculator
Action Input: 125 * 8 + 37
Observation: 125 * 8 + 37 = 1037
Thought:I now know the final answer
Final Answer: 1037

> Finished chain.
✅ Answer: 1037


In [3]:

print("\n─── Agent Test 2: Word Counter ───")
try:
    result = agent.run("How many words are in the sentence: 'LangChain makes building LLM apps easy'?")
    print(f"✅ Answer: {result}")
except Exception as e:
    print(f"❌ {e}")


─── Agent Test 2: Word Counter ───


> Entering new AgentExecutor chain...
❌ Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 6.159479698s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId':

In [4]:

print("\n─── Agent Test 3: Reverse Text ───")
try:
    result = agent.run("Reverse this sentence: 'Python is the best coding language in the world'.")
    print(f"✅ Answer: {result}")
except Exception as e:
    print(f"❌ {e}")


─── Agent Test 3: Reverse Text ───


> Entering new AgentExecutor chain...
Action: reverse_text
Action Input: Python is the best coding language in the world
Observation: dlrow eht ni egaugnal gnidoc tseb eht si nohtyP
Thought:I now know the final answer
Final Answer: dlrow eht ni egaugnal gnidoc tseb eht si nohtyP

> Finished chain.
✅ Answer: dlrow eht ni egaugnal gnidoc tseb eht si nohtyP


In [5]:
# ── Core Abstraction 1: MODELS ────────────────────────────────────────────
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

print("📦 PART A – LangChain Core Abstractions")
print("=" * 55)
print("\n── Model abstraction ──")

# Chat Model: takes a list of messages, returns AIMessage
chat_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    google_api_key=gemini_api_key,
)

# Direct invocation with structured messages
messages = [
    SystemMessage(content="You are a concise AI tutor."),
    HumanMessage(content="In one sentence, what is LangChain?"),
]
response = chat_model.invoke(messages)
print(f"Response type : {type(response).__name__}")
print(f"Response text : {response.content.strip()}")

📦 PART A – LangChain Core Abstractions

── Model abstraction ──
Response type : AIMessage
Response text : LangChain is a framework designed to simplify the development of applications powered by large language models by providing tools to chain together various components.


In [6]:
# ── Core Abstraction 2: PROMPTS ───────────────────────────────────────────
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

print("\n── PromptTemplate (string-based) ──")
pt = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in exactly two sentences.",
)
formatted = pt.format(topic="vector embeddings")
print("Formatted prompt:\n", formatted)

print("\n── ChatPromptTemplate (message-based) ──")
chat_pt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}."),
    ("human", "Tell me about {subject} briefly."),
])
formatted_msgs = chat_pt.format_messages(role="data scientist", subject="gradient descent")
for msg in formatted_msgs:
    print(f"  [{msg.__class__.__name__}]: {msg.content}")


── PromptTemplate (string-based) ──
Formatted prompt:
 Explain vector embeddings in exactly two sentences.

── ChatPromptTemplate (message-based) ──
  [SystemMessage]: You are a data scientist.
  [HumanMessage]: Tell me about gradient descent briefly.


In [7]:
# ── Core Abstraction 3: OUTPUT PARSERS + Hello-World ─────────────────────
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

print("\n── Hello-World with LCEL (prompt | model | parser) ──")

# Build a minimal chain using the pipe operator
hello_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    ("human", "{question}"),
])
parser = StrOutputParser() # just returns the raw text response without parsing

# The LCEL chain: prompt → model → parser
hello_chain = hello_prompt | chat_model | parser

result = hello_chain.invoke({"question": "What are the three core abstractions of LangChain LCEL?"})
print("Hello-World result:")
print(result)


── Hello-World with LCEL (prompt | model | parser) ──
Hello-World result:
The three core abstractions of LangChain LCEL are:

1.  **Runnables**: The fundamental building blocks (e.g., models, chains, tools).
2.  **RunnableSequence**: For composing runnables sequentially.
3.  **RunnableParallel**: For composing runnables in parallel.


In [8]:

# ── JSON output parser ────────────────────────────────────────────────────
print("\n── JSON Output Parser ──")
json_prompt = ChatPromptTemplate.from_messages([
    ("human", 'Return a JSON object with keys "name" and "capital" for the country: {country}. Return ONLY the JSON.'),
])
json_chain = json_prompt | chat_model | JsonOutputParser()
json_result = json_chain.invoke({"country": "Japan"})
print(f"Parsed JSON : {json_result}")
print(f"Capital     : {json_result.get('capital')}")


── JSON Output Parser ──
Parsed JSON : {'name': 'Japan', 'capital': 'Tokyo'}
Capital     : Tokyo


In [9]:
# ── Multi-step LCEL pipeline ──────────────────────────────────────────────
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("⛓️  PART B – Chains / Runnables")
print("=" * 55)

# Step 1: Generate a 3-point outline
outline_prompt = ChatPromptTemplate.from_messages([
    ("human", "Create a 3-point bullet outline for a blog post about: {topic}"),
])
outline_chain = outline_prompt | chat_model | StrOutputParser()

# Step 2: Expand the outline into an introduction paragraph
expand_prompt = ChatPromptTemplate.from_messages([
    ("human", "Using this outline:\n{outline}\n\nWrite a short 2-sentence introduction for the blog post."),
])
expand_chain = expand_prompt | chat_model | StrOutputParser()

topic = "the impact of AI on software development"
print(f"Topic: {topic}\n")

outline_result = outline_chain.invoke({"topic": topic})
print("── Step 1: Outline ──")
print(outline_result)

intro_result = expand_chain.invoke({"outline": outline_result})
print("\n── Step 2: Introduction ──")
print(intro_result)

⛓️  PART B – Chains / Runnables
Topic: the impact of AI on software development

── Step 1: Outline ──
Here's a 3-point bullet outline for a blog post on the impact of AI on software development:

*   **1. Automating & Accelerating the Development Lifecycle:**
    *   AI-powered code generation (e.g., Copilot, Tabnine) for faster prototyping and boilerplate.
    *   Enhanced testing and debugging, identifying errors and vulnerabilities more efficiently.
    *   Automated documentation, refactoring, and code review processes.

*   **2. Transforming the Developer's Role and Skillset:**
    *   Shifting focus from repetitive coding to higher-level design, architecture, and problem-solving.
    *   Increased demand for "prompt engineering" and AI model interaction skills.
    *   Emphasis on ethical considerations, bias detection, and ensuring AI-generated code quality.

*   **3. Unlocking New Possibilities and Navigating Emerging Challenges:**
    *   Enabling the creation of more intelli

In [10]:
# ── RunnableParallel – run multiple chains side-by-side ──────────────────
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("── RunnableParallel: pros & cons at once ──")

pros_prompt = ChatPromptTemplate.from_messages([
    ("human", "List 2 pros of {technology} in one sentence each."),
])
cons_prompt = ChatPromptTemplate.from_messages([
    ("human", "List 2 cons of {technology} in one sentence each."),
])

parallel_chain = RunnableParallel(
    pros=pros_prompt | chat_model | StrOutputParser(),
    cons=cons_prompt | chat_model | StrOutputParser(),
)

result = parallel_chain.invoke({"technology": "Large Language Models"})
print("PROS:\n", result["pros"])
print("\nCONS:\n", result["cons"])

── RunnableParallel: pros & cons at once ──
PROS:
 1.  Large Language Models can efficiently process and summarize vast amounts of information, making complex data more accessible and understandable.
2.  They can automate various text-based tasks, significantly improving productivity and efficiency across many industries.

CONS:
 1.  Large Language Models can "hallucinate" or confidently generate factually incorrect information, making their outputs unreliable without verification.
2.  They often perpetuate and amplify biases present in their training data, leading to unfair, discriminatory, or stereotypical responses.


In [11]:
# ── RunnableBranch – conditional routing (branching lite) ─────────────────
from langchain_core.runnables import RunnableBranch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("── RunnableBranch: route to specialist chain ──")

# Three specialist chains
tech_chain = (
    ChatPromptTemplate.from_messages([("human", "Answer this tech question concisely: {question}")])
    | chat_model | StrOutputParser()
)
legal_chain = (
    ChatPromptTemplate.from_messages([("human", "Answer this legal question in simple terms: {question}")])
    | chat_model | StrOutputParser()
)
default_chain = (
    ChatPromptTemplate.from_messages([("human", "Answer this question helpfully: {question}")])
    | chat_model | StrOutputParser()
)

branch = RunnableBranch(
    (lambda x: any(kw in x["question"].lower() for kw in ["python", "code", "programming"]), tech_chain),
    (lambda x: any(kw in x["question"].lower() for kw in ["law", "legal", "copyright"]),      legal_chain),
    default_chain,   # fallback
)

test_questions = [
    {"question": "How do Python decorators work?"},
    {"question": "What is copyright law?"},
    {"question": "What is the tallest mountain in the world?"},
]

for q in test_questions:
    ans = branch.invoke(q)
    print(f"\n❓ {q['question']}")
    print(f"💬 {ans[:180].strip()}...")

── RunnableBranch: route to specialist chain ──

❓ How do Python decorators work?
💬 Python decorators are functions that take another function as an argument, modify or extend its behavior, and then return a *new* function (often a wrapper). The `@decorator_name`...

❓ What is copyright law?
💬 In simple terms, **copyright law is a special legal protection for things people create.**

Imagine you've written a book, composed a song, painted a picture, or even developed a c...

❓ What is the tallest mountain in the world?
💬 The tallest mountain in the world, by far the most widely accepted and common definition, is **Mount Everest**.

Here are the details:

*   **Height:** Approximately **8,848.86 met...


In [12]:
# ── Tool Calling – define tools with @tool decorator ─────────────────────
from langchain_core.tools import tool
import math
import datetime

print("🛠️  PART C – Tools & Agents")
print("=" * 55)
print("\n── Defining tools with @tool ──")

@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together and return the result."""
    return a + b

@tool
def get_current_date() -> str:
    """Return today's date in YYYY-MM-DD format."""
    return datetime.date.today().isoformat()

@tool
def calculate_square_root(number: float) -> float:
    """Calculate the square root of a non-negative number."""
    if number < 0:
        return "Error: cannot take square root of a negative number."
    return round(math.sqrt(number), 4)

@tool
def word_count(text: str) -> int:
    """Count the number of words in a given text string."""
    return len(text.split())

tools = [add_numbers, get_current_date, calculate_square_root, word_count]

print("Registered tools:")
for t in tools:
    print(f"  • {t.name:25s} – {t.description}")

print("\n── Manual tool invocation (unit-test style) ──")
print("  add_numbers(7, 3)              →", add_numbers.invoke({"a": 7, "b": 3}))
print("  get_current_date()             →", get_current_date.invoke({}))
print("  calculate_square_root(144)     →", calculate_square_root.invoke({"number": 144}))
print("  word_count('hello world foo')  →", word_count.invoke({"text": "hello world foo"}))

🛠️  PART C – Tools & Agents

── Defining tools with @tool ──
Registered tools:
  • add_numbers               – Add two numbers together and return the result.
  • get_current_date          – Return today's date in YYYY-MM-DD format.
  • calculate_square_root     – Calculate the square root of a non-negative number.
  • word_count                – Count the number of words in a given text string.

── Manual tool invocation (unit-test style) ──
  add_numbers(7, 3)              → 10.0
  get_current_date()             → 2026-03-11
  calculate_square_root(144)     → 12.0
  word_count('hello world foo')  → 3


In [13]:
# ── Function Interface – bind_tools: LLM picks a tool ────────────────────
from langchain_core.messages import HumanMessage

print("── bind_tools: LLM decides which tool to call ──\n")

# Bind tools so the model can generate tool-call requests
llm_with_tools = chat_model.bind_tools(tools)

test_queries = [
    "What is the square root of 256?",
    "What is today's date?",
    "How many words are in the sentence: The quick brown fox jumps over the lazy dog?",
]

tool_map = {t.name: t for t in tools}

for query in test_queries:
    response = llm_with_tools.invoke([HumanMessage(content=query)])
    print(f"Query: {query}")

    if response.tool_calls:
        tc = response.tool_calls[0]
        print(f"  → Tool selected : {tc['name']}")
        print(f"  → Tool args     : {tc['args']}")
        # Execute the chosen tool
        tool_result = tool_map[tc["name"]].invoke(tc["args"])
        print(f"  → Tool result   : {tool_result}")
    else:
        print(f"  → Direct answer : {response.content.strip()}")
    print()

── bind_tools: LLM decides which tool to call ──

Query: What is the square root of 256?
  → Tool selected : calculate_square_root
  → Tool args     : {'number': 256}
  → Tool result   : 16.0

Query: What is today's date?
  → Tool selected : get_current_date
  → Tool args     : {}
  → Tool result   : 2026-03-11

Query: How many words are in the sentence: The quick brown fox jumps over the lazy dog?
  → Tool selected : word_count
  → Tool args     : {'text': 'The quick brown fox jumps over the lazy dog'}
  → Tool result   : 9



In [14]:
# ── Agent Loop – manual ReAct-style loop (Think → Act → Observe) ──────────
from langchain_core.messages import HumanMessage, ToolMessage

print("── Manual ReAct Agent Loop ──")

def run_agent(user_query: str, max_steps: int = 6) -> str:
    """
    Minimal ReAct loop:
      1. Send messages to LLM
      2. If LLM returns tool_calls → execute each tool, append ToolMessage(s)
      3. If LLM returns text        → final answer, stop
    """
    messages = [HumanMessage(content=user_query)]
    print(f"\n{'─'*50}")
    print(f"👤 User : {user_query}")

    for step in range(1, max_steps + 1):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            # LLM produced a final text answer
            final_content = response.content
            if isinstance(final_content, list):
                final_content = " ".join(str(item) for item in final_content)
            print(f"✅ Final : {final_content.strip()}")
            return final_content

        # Execute each requested tool call
        for tc in response.tool_calls:
            print(f"  [step {step}] tool={tc['name']}  args={tc['args']}")
            result = tool_map[tc["name"]].invoke(tc["args"])
            print(f"  [step {step}] result={result}")
            messages.append(
                ToolMessage(content=str(result), tool_call_id=tc["id"])
            )

    return "Max steps reached."

# ── Test the agent ─────────────────────────────────────────────────────────
run_agent("What is 47 plus 89?")
run_agent("What is today's date and also the square root of 225?")

── Manual ReAct Agent Loop ──

──────────────────────────────────────────────────
👤 User : What is 47 plus 89?
  [step 1] tool=add_numbers  args={'a': 47, 'b': 89}
  [step 1] result=136.0
✅ Final : 47 + 89 = 136.

──────────────────────────────────────────────────
👤 User : What is today's date and also the square root of 225?
  [step 1] tool=get_current_date  args={}
  [step 1] result=2026-03-11
  [step 2] tool=calculate_square_root  args={'number': 225}
  [step 2] result=15.0
✅ Final : Today's date is 2026-03-11 and the square root of 225 is 15.


"Today's date is 2026-03-11 and the square root of 225 is 15."